### <font color="skyblue">Face Dataset Builder</font>

This notebook reproduces `facenetPytorch_build_dataset.py`:

- Loads the Haar cascade face detector.
- Opens the webcam and detects faces.
- Draws green rectangles around detected faces.
- Press `k` to save the original frame.
- Press `q` to stop the camera loop.
<hr>

### <font color="skyblue"> 1. Environment Setup and Dependency Imports

In [ ]:
import time
import cv2
import imutils
from pathlib import Path
from imutils.video import VideoStream

### <font color="skyblue"> 2. Script Configuration and Constants

In [2]:
# Paths are based on this notebook's folder, so the XML file is found regardless of cwd.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "haarcascade_frontalface_default.xml").exists():
    NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd() / "Facenet_Pytorch"

CASCADE_PATH = NOTEBOOK_DIR / "haarcascade_frontalface_default.xml"
DATASET_ROOT = NOTEBOOK_DIR / "facenet_dataset"
CAMERA_INDEX = 0
FRAME_WIDTH = 400


### <font color="skyblue"> 3. Function and Class Definitions from the `.py` File </font>

The source script does not define reusable functions or classes. Its executable statements are preserved in the ordered cells below.

### <font color="skyblue"> 4. Input Loading and Preprocessing Pipeline

In [3]:
detector = cv2.CascadeClassifier(str(CASCADE_PATH))
if detector.empty():
    raise FileNotFoundError(f"Cannot load cascade file: {CASCADE_PATH}")

print(f"Loaded cascade: {CASCADE_PATH}")


Loaded cascade: c:\Users\CCWANG\OneDrive\PythonSurfaceBook\Shallow_ML\Facenet_Pytorch\haarcascade_frontalface_default.xml


### <font color="skyblue"> 5. Main Execution Flow</font>

In [4]:
username = input("Enter the userID for the face: ").strip()
if not username:
    raise ValueError("The user ID cannot be empty.")

output_path = DATASET_ROOT / username
output_path.mkdir(parents=True, exist_ok=True)

print(f"Images will be saved to: {output_path}")


Images will be saved to: c:\Users\CCWANG\OneDrive\PythonSurfaceBook\Shallow_ML\Facenet_Pytorch\facenet_dataset\wang


In [5]:
print("[INFO] starting video stream...")
vs = VideoStream(src=CAMERA_INDEX).start()
time.sleep(2.0)
total = 0

try:
    while True:
        frame = vs.read()
        if frame is None:
            print("[WARN] Could not read a frame from the camera.")
            break

        original_frame = frame.copy()
        resized_frame = imutils.resize(frame, width=FRAME_WIDTH)
        gray_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2GRAY)

        rects = detector.detectMultiScale(
            gray_frame,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30),
        )

        for (x, y, width, height) in rects:
            cv2.rectangle(
                resized_frame,
                (x, y),
                (x + width, y + height),
                (0, 255, 0),
                2,
            )

        cv2.imshow("Frame", resized_frame)
        key = cv2.waitKey(1) & 0xFF

        if key == ord("k"):
            image_path = output_path / f"{total:05d}.png"
            cv2.imwrite(str(image_path), original_frame)
            total += 1
            print(f"[INFO] saved {image_path}")
        elif key == ord("q"):
            break
finally:
    print(f"[INFO] {total} face images stored")
    print("[INFO] cleaning up...")
    vs.stop()
    cv2.destroyAllWindows()


[INFO] starting video stream...
[INFO] 0 face images stored
[INFO] cleaning up...


### <font color="skyblue"> 6. Output Generation and Persistence</font>

Captured original frames are saved as five-digit PNG files in `facenet_dataset/<userID>/`, matching the script's naming convention.

### <font color="skyblue"> 7. Quick Validation Cells</font>

In [ ]:
assert CASCADE_PATH.is_file(), f"Missing cascade file: {CASCADE_PATH}"
assert not detector.empty(), "The Haar cascade detector did not load."
assert output_path.is_dir(), f"Missing output directory: {output_path}"

saved_images = sorted(output_path.glob("*.png"))
assert total == len(saved_images), (
    f"Saved-image count mismatch: total={total}, files={len(saved_images)}"
)
print(f"Validation passed: {len(saved_images)} PNG file(s) in {output_path}")
